# Feature Extraction: TF-IDF and Text Features

**Source:** LinkedIn Learning Course  
**Topic:** Feature Engineering - Text Feature Extraction  
**Status:** Exploratory notebook for learning text feature extraction patterns

## Overview

This notebook demonstrates text feature extraction techniques, specifically **TF-IDF (Term Frequency-Inverse Document Frequency)**, a fundamental method for converting text into numerical features for machine learning.

## Key Concepts Covered

### TF-IDF (Term Frequency-Inverse Document Frequency)

**What it measures:** How important a word is to a document in a collection

**Two components:**
1. **TF (Term Frequency)**: How often a word appears in a document
   - High TF = word is common in this specific document
2. **IDF (Inverse Document Frequency)**: How rare a word is across all documents
   - High IDF = word is rare, therefore more distinctive

**Formula:** TF-IDF = TF × IDF

**Result:** Common words like "the", "and" get low scores. Distinctive words get high scores.

### Applications
- **Text Classification**: Spam detection, sentiment analysis, topic categorization
- **Document Search**: Rank documents by relevance to query
- **Recommendation Systems**: Find similar documents
- **Feature Engineering**: Convert text to numerical features for ML models

## Dataset Context

Uses EPA Fuel Economy data where text features might include:
- `eng_dscr` (engine description) - free text field
- `trans_dscr` (transmission description) - free text field
- `model` (vehicle model names) - text feature

These text fields contain valuable information (turbo, hybrid, automatic) that needs numerical encoding.

## Notes for Future Revision

**Security & Data Management:**
- ✅ Loads data from public EPA API (no credentials)
- ✅ No hardcoded secrets
- ✅ Self-contained data loading

**Key Learnings:**
- **Stop words matter**: Remove common words ("the", "and", "is") to reduce noise
- **Max features**: Limit vocabulary size to most important terms (memory/performance)
- **N-grams**: Capture phrases ("manual transmission" vs separate "manual" and "transmission")
- **Sparse matrices**: TF-IDF produces sparse matrices (most values are zero) - use scipy sparse format

**Production Patterns:**
- Fit TfidfVectorizer on training data only (prevent leakage)
- Save fitted vectorizer (pickle) for inference
- Use `max_features` to cap vocabulary (prevent memory issues)
- Consider hashing vectorizer for very large vocabularies
- Monitor vocabulary drift in production (new terms appearing)

**Comparison to Modern Approaches:**
- **TF-IDF (classical)**: Simple, interpretable, fast, no training needed
- **Word2Vec/GloVe**: Captures semantic meaning, context-aware
- **Transformers (BERT)**: State-of-art, but heavy computational cost
- **TF-IDF still valuable**: Baseline, explainability, low-resource scenarios

**Related to AI Portfolio:**
- Connects to notes/03-ai embedding chapters (TF-IDF is a simple embedding method)
- Foundation for understanding modern embeddings (sentence-transformers, OpenAI embeddings)
- RAG systems: TF-IDF can be used for lightweight document retrieval before LLM generation

**Improvements Needed:**
- [ ] Add comparison: TF-IDF vs modern embeddings (sentence-transformers)
- [ ] Show classification example with TF-IDF features
- [ ] Add n-gram examples (bigrams, trigrams)
- [ ] Visualize vocabulary and top TF-IDF scores
- [ ] Document memory usage for different max_features settings
- [ ] Add cosine similarity for document comparison
- [ ] Show proper train/test split with fitted vectorizer

**Dependencies:**
- pandas, numpy
- scikit-learn (TfidfVectorizer, feature_extraction.text)
- EPA vehicles API (public)

**Common Pitfalls to Avoid:**
- ❌ Fitting vectorizer on entire dataset (leaks test vocabulary into training)
- ❌ Not removing stop words (noise in features)
- ❌ Setting max_features too high (memory explosion)
- ❌ Ignoring rare terms (can be informative, but also noisy)
- ❌ Not saving fitted vectorizer (can't reproduce features in production)

**Bridge to Modern AI:**
- TF-IDF → Sparse embeddings (scikit-learn)
- Word2Vec → Dense embeddings (gensim)
- Sentence-Transformers → Contextual embeddings (HuggingFace)
- OpenAI Embeddings → Production-grade embeddings (API)

This notebook teaches classical NLP feature extraction - foundation for understanding modern embedding techniques used in RAG systems and LLM applications.

---

**TFIDF**

*Frequency-inverse document frequency (TFIDF)* is a numerical statistic that is intended to reflect how important a word is to a document in a collection or corpus. It is often used as a feature for text classification.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import pandas as pd
import numpy as np
from utils import (
    load_fuel_economy_data,
    debug_transformer,
    combine_str_cols_transformer,
    TextPipeline,
    CAT_COLS, LOW_CARDINALITY_COLS, HIGH_CARDINALITY_COLS,
    make_preprocessor,
)

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# create X and y
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# full pipeline: preprocessing (TF-IDF text + OHE + target encoding) → debug → scaling → regression
pipeline = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('debug', FunctionTransformer(debug_transformer, kw_args={'name': 'tmp_X'})),
    ('lr', LinearRegression()),
])

pipeline.fit(X_train, y_train)
print("R² on test set:", pipeline.score(X_test, y_test))